In [ ]:
# install dependencies
%pip install anthropic python-dotenv

In [1]:
#load environment variables

from dotenv import load_dotenv
load_dotenv()

True

In [2]:
# create client and set model
from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-4-6"

In [3]:
def add_user_message(messages, content):
    messages.append({"role": "user", "content": content})
    return messages

def add_assistant_message(messages, content):
    messages.append({"role": "assistant", "content": content})
    return messages

def send_message(messages, system=None, stop_sequences=None, json_output=False):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
    }
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    if json_output:
        params["output_config"] = {"format": {"type": "json_object"}}

    message = client.messages.create(**params)
    return message


In [ ]:
messages = []
response = client.messages.create(
    model="claude-sonnet-4-5",
    max_tokens=1000,
    messages=[
        {"role": "user", "content": "Generate a very short EventBridge rule"}
    ],
    output_config={
        "format": {
            "type": "json_schema",
            "schema": {
                "type": "object",
                "properties": {
                    "source": {"type": "array", "items": {"type": "string"}},
                    "detail-type": {"type": "array", "items": {"type": "string"}},
                    "detail": {
                        "type": "object",
                        "additionalProperties": False  # <-- Added to the nested object
                    }
                },
                "required": ["source", "detail-type", "detail"],
                "additionalProperties": False  # <-- Added to the root object
            }
        }
    }
)

data = json.loads(response.content[0].text)
print(data)


{'source': ['custom.myapp'], 'detail-type': ['Order Placed'], 'detail': {}}


In [16]:
# prefill and stop sequence is old approach now its deprecated in new models and will trhow errors if we use for json output
# Instead, use the output_config with json_schema to get structured JSON output
messages = []
add_user_message(
    messages, 
    "Generate a very short EventBridge rule. Output ONLY the raw JSON. Do not include markdown code fences (like ```json), introduction, or explanations."
)

response = client.messages.create(
    model="claude-sonnet-4-5",
    max_tokens=1000,
    messages=messages
)

response

Message(id='msg_011Cf5p5NwpzafTqR9bcJ1md', container=None, content=[TextBlock(citations=None, text='{\n  "Name": "DailyLambdaTrigger",\n  "ScheduleExpression": "rate(1 day)",\n  "State": "ENABLED",\n  "Targets": [\n    {\n      "Arn": "arn:aws:lambda:us-east-1:123456789012:function:MyFunction",\n      "Id": "1"\n    }\n  ]\n}', type='text')], model='claude-sonnet-4-5-20250929', role='assistant', stop_details=None, stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=41, output_tokens=102, output_tokens_details=None, server_tool_use=None, service_tier='standard'))